# Gemma Edu-Agent: Facilitando la Personalización a Escala

*Generación de andamiaje conceptual mediante modelos de frontera para docentes en entornos de alta densidad.*

## 1. El Problema: El Cuello de Botella Pedagógico
En América Latina, el sistema educativo público enfrenta aulas de más de 40 alumnos por docente. Explicar conceptos técnicos abstractos de física o matemáticas exige horas de re-explicaciones individualizadas. La personalización pedagógica a menudo resulta inviable por falta de tiempo. 

Asimismo, el uso no guiado de IA generativa presenta un reto: cuando los alumnos reciben resoluciones directas de sus tareas, el esfuerzo cognitivo disminuye. Nuestra hipótesis es que los docentes necesitan una herramienta que asista en la construcción de andamiaje mental, no un solucionador de ejercicios.

## 2. Solución Implementada: Analogías Estructuradas
**Gemma Edu-Agent** es un prototipo diseñado para ayudar al profesor. El MVP toma conceptos extraídos de literatura abierta (ej. *OpenStax Physics*) y asiste en la creación de analogías basadas en los intereses del estudiante (videojuegos, deportes, etc.).

El sistema no resuelve problemas numéricos; en su lugar, mapea explícitamente las reglas del concepto técnico a las reglas del dominio de interés del alumno, fomentando la comprensión relacional.

## 3. Arquitectura del Prototipo
La arquitectura actual demuestra el flujo de extremo a extremo utilizando los siguientes componentes:

1. **Recuperación Semántica Ligera (RAG):** El MVP utiliza el modelo `models/embedding-001` de Google para generar embeddings de fragmentos de texto preseleccionados. La recuperación se realiza mediante un cálculo de similitud coseno en memoria utilizando `numpy`. Esta decisión priorizó la estabilidad y reproducibilidad en entornos locales durante el hackathon frente a bases vectoriales externas.
2. **Generación con Salidas Estructuradas (Structured Outputs):** Para controlar el formato de la respuesta, el agente se integra mediante la API generativa de Google forzando un esquema Pydantic. Esto restringe al modelo para emitir un documento JSON validable con campos específicos:
   * `technical_concept` y `source_citation`
   * `student_interest`
   * `conceptual_analogy`
   * `mapping_matrix` (Tabla relacional)
   * `verification_question` (Enfocada en transferencia, no en memoria)

## 4. Decisiones de Ingeniería para el MVP
* **Recuperación vs. Entrenamiento:** Se eligió un enfoque RAG en memoria en lugar de fine-tuning para reducir la latencia de implementación y asegurar que las respuestas se basaran explícitamente en el material de OpenStax proporcionado.
* **Reducción de Dependencias:** El reemplazo de bases de datos vectoriales complejas por una solución nativa basada en `numpy` permitió que el desarrollo y la ejecución del servidor Streamlit se mantuvieran estables en un entorno local (Windows) durante el límite de tiempo del evento.

## Enlaces del Proyecto
* **Demo en Vivo (Notebook Ejecutable):** [AÑADIR_URL]
* **Repositorio de Código Público:** https://github.com/ricardomartinezsau-jpg/hackday
* **Video de Demostración:** [AÑADIR_URL_YOUTUBE]



# Gemma Edu-Agent (Demo interactiva)

Este cuaderno permite ejecutar el pipeline del Agente sin necesidad de montar un servidor Streamlit. Funciona directamente en Google Colab o Kaggle Notebooks.

In [ ]:
!pip install google-generativeai pydantic numpy python-dotenv -q

In [ ]:
import os
import google.generativeai as genai
from pydantic import BaseModel, Field
from typing import List
import json
import numpy as np

# 1. Configura tu API Key de Google AI Studio aquí:
os.environ['GEMINI_API_KEY'] = 'TU_API_KEY_AQUI'
genai.configure(api_key=os.environ['GEMINI_API_KEY'])

In [ ]:
# 2. Simulamos la recuperación de texto (RAG)
texto_recuperado = "La ley de conservación del momento lineal establece que si la fuerza neta externa que actúa sobre un sistema de partículas es cero, el momento lineal total del sistema permanece constante. m1*v1_i + m2*v2_i = m1*v1_f + m2*v2_f"

# 3. Definimos los intereses del alumno
interes_alumno = "Jugar billar con sus amigos"

In [ ]:
# 4. Definición del Esquema (Pydantic)
class MappingObject(BaseModel):
    academico: str
    analogico: str

class PedagogicalAnalogy(BaseModel):
    technical_concept: str
    source_citation: str
    student_interest: str
    conceptual_analogy: str
    mapping_matrix: List[MappingObject]
    verification_question: str

# 5. Ejecución del Agente con Function Calling
model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    system_instruction="Eres un diseñador instruccional riguroso. Genera una analogía estructurada y precisa basándote estrictamente en el texto fuente proporcionado."
)

prompt = f"Contexto:\n{texto_recuperado}\n\nInterés del Alumno:\n{interes_alumno}\n\nGenera una analogía pedagógica estructurada."

try:
    response = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            response_schema=PedagogicalAnalogy,
        ),
    )
    print(json.dumps(json.loads(response.text), indent=2, ensure_ascii=False))
except Exception as e:
    print("Error:", e)